In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import os
warnings.filterwarnings('ignore')

# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = ''

# Load Datasets

In [ ]:
playlist = pd.read_parquet(os.path.join(BASE_DIR, 'playlist_cleaned.parquet'))
track = pd.read_parquet(os.path.join(BASE_DIR, 'track_final.parquet')) 

# Mask last 10 songs in each playlist

As we will be recommending tracks to the playlist, we would need to mask 10 tracks, in order for us to evaluate the performance of the model later on

### Filtering Playlists

We will remove playlists that have < 20 tracks in the playlist, as there are too few tracks.

In [120]:
original_length = len(playlist)
playlist = playlist[playlist['num_tracks'] >= 20]
removed_rows = original_length - len(playlist)

print(f"Number of rows removed: {removed_rows}")

Number of rows removed: 1496


### Removing last 10 tracks from each playlist

Store tracks inside playlist as list of track_idx contained in the playlist for convenience and speed in later tasks

In [121]:
# Convert the string to a dictionary and extract the list of track indices
playlist['track_idx_list'] = playlist['tracks'].apply(lambda track_str: 
    list(ast.literal_eval(track_str.replace("'", "\"")).values()) 
    if isinstance(track_str, str) else list(track_str.values())
)

# Output the result
playlist[['track_idx_list']].head()

,track_idx_list
18539,"[3689, 207774, 194775, 135193, 218011, 37844, ..."
18511,"[160375, 131195, 164629, 147280, 193891, 17077..."
18507,"[104436, 229428, 25968, 186871, 81592, 15947, ..."
6210,"[244173, 210510, 9349, 224202, 147251, 35498, ..."
6245,"[194410, 10513, 62267, 196463, 14164, 13405, 1..."


We will store the 10 tracks separately in a column named `tracks_to_predict`

In [122]:
playlist[['track_idx_list', 'tracks_to_predict']] = playlist['track_idx_list'].apply(
    lambda x: pd.Series(
        (x.tolist()[:-10], x.tolist()[-10:]) if isinstance(x, np.ndarray) else
        (ast.literal_eval(x)[:-10], ast.literal_eval(x)[-10:]) if isinstance(x, str) else
        (x[:-10], x[-10:]) if isinstance(x, list) else
        ([], [])
    )
)

### Adjust `num_tracks` and `num_edits` columns

Create `avg_tracks_per_edit` column

In [123]:
playlist["avg_tracks_per_edit"] = playlist["num_tracks"] / playlist["num_edits"]
playlist["avg_tracks_per_edit"] = playlist["avg_tracks_per_edit"].round(3)

Since number of tracks (in our 'seen' playlist) have been reduced by 10, we will deduct 10 from `num_tracks`

In [124]:
# Deduct 10 from num_tracks
playlist['num_tracks'] = playlist['num_tracks'] - 10

We will scale the `num_edits` accordingly to ensure that the average number of tracks added per edit is still the same

In [125]:
# Recalculate num_edits to maintain avg_tracks_per_edit
playlist['num_edits'] = (playlist['num_tracks'] / playlist['avg_tracks_per_edit']).round().astype(int)

# Ensure no negative values
playlist['num_tracks'] = playlist['num_tracks'].clip(lower=0)
playlist['num_edits'] = playlist['num_edits'].clip(lower=0)

# Track Weight Assignment
Based on `num_edits` and `num_tracks`

We hypothesize that the more recently added tracks to the playlist is a better representation of the user's preference. As such, we would be assigning weights for each tracks in the playlist. 
<br>
<br>
The creation of the weights follows this process:
<br>
<br>
1. **Triangular Number Formula for Total Weight Sum**:
   To compute the total weight sum for the edits, the triangular number formula is used:
   $$
   \text{Total Weight Sum} = \sum_{i=1}^{n} i = \frac{n(n+1)}{2}
   $$
   where $n$ is the number of edits (`num_edits`). This sum represents the cumulative weight over all edits.
<br>
<br>
2. **Track Distribution**:

   - **Base Tracks per Edit**: Each edit is assigned an equal number of tracks, computed as:
     $$
     \text{Base tracks per edit} = \left\lfloor \frac{\text{num_tracks}}{\text{num_edits}} \right\rfloor
     $$
   - **Extra Tracks**: The remaining tracks, calculated as:
     $$
     \text{extra tracks} = {\text{num_tracks}}\;  \% \;{\text{num_edits}}
     $$
     These extra tracks are distributed by adding 1 track to the most recent edit.
<br>
<br>
3. **Weight per Edit**:
   The weight for each edit is calculated as:
   $$
    \text{edit_weight}[i] = \frac{i + 1}{\text{Total Weight Sum}}
   $$
   where $i$ is the index of the edit (starting from 0).
<br>
<br>
4. **Weight Assignment to Tracks**:
   For each edit, the weight is distributed among the tracks:
   $$
    \text{track_weight} = \frac{\text{edit_weight}[i]}{\text{batch_size}}
   $$
   where `batch_size` is the number of tracks in the current edit. Each track gets a weight that is rounded to 5 decimal places.
<br>
<br>
5. **Result**:
   The final output is a list of tuples where each tuple contains a track and its corresponding weight. This list is added as a new column `tracks_weights_5dp` in the playlist DataFrame.


Example

- A playlist contains 16 tracks, and num_edit is 3.

- Using the Triangular Number Formula, the total_weight_sum is 6, which would be the denominator for the weight of each batch.

- The tracks are distributed in this way: The first edit had 5 songs, second edit had 5 songs, and the third edit had 6 songs. 

    - The weight for the first edit batch is $\frac{1}{6}$, since the index of this edit is 1

    - The weight for the second edit batch is $\frac{2}{6}$, since the index of this edit is 2

    - The weight for the third edit batch is $\frac{3}{6}$, since the index of this edit is 3

- For each individual tracks in the batch, we would have to do a further division by the number of tracks in the batch

    - The individual weights for the tracks in the first batch (with 5 tracks) is $\frac{1}{6\times5} = \frac{1}{30}$

    - The individual weights for the tracks in the second batch (with 5 tracks) is $\frac{2}{6\times5} = \frac{1}{15}$

    - The individual weights for the tracks in the third batch (with 6 tracks) is $\frac{3}{6\times6} = \frac{1}{12}$

- Summing up the weights for each individual tracks, $(5 \times \frac{1}{30}) + (5 \times \frac{1}{15}) + (6 \times \frac{1}{12}) = 1$

In [126]:
# Check for rows where num_edits > num_tracks
rows_with_more_edits_than_tracks = playlist[playlist['num_edits'] > playlist['num_tracks']]

# Print out the rows that satisfy the condition (this is optional, for inspection)
print("Before changes:")
print(rows_with_more_edits_than_tracks[['playlist_idx', 'num_edits', 'num_tracks', 'avg_tracks_per_edit']].head())

# Store the original index (track_idx) for later retrieval
track_idx_before_change = rows_with_more_edits_than_tracks['playlist_idx'].values

# Modify 'avg_tracks_per_edit' for the rows where num_edits > num_tracks
playlist.loc[playlist['num_edits'] > playlist['num_tracks'], 'avg_tracks_per_edit'] = 1

# Now adjust num_edits column by subtracting 1 for the rows where num_edits > num_tracks
playlist['num_edits'] = playlist.apply(
    lambda row: row['num_edits'] - 1 if row['num_edits'] > row['num_tracks'] else row['num_edits'],
    axis=1
)

# After the changes, retrieve the rows using the stored track_idx
rows_with_more_edits_than_tracks_after = playlist[playlist['playlist_idx'].isin(track_idx_before_change)]

# Print out the rows after changes for comparison
print("After changes:")
print(rows_with_more_edits_than_tracks_after[['playlist_idx', 'num_edits', 'num_tracks', 'avg_tracks_per_edit']].head())

Before changes:
       playlist_idx  num_edits  num_tracks  avg_tracks_per_edit
6425           9644         13          12                0.957
16925         11494         11          10                0.952
After changes:
       playlist_idx  num_edits  num_tracks  avg_tracks_per_edit
6425           9644         12          12                  1.0
16925         11494         10          10                  1.0


Code for the Algorithm

In [127]:
track_weights_list = []

for index, row in playlist.iterrows():
    num_tracks = row['num_tracks']
    num_edits = row['num_edits']
    tracks = ast.literal_eval(row['tracks'])  # Convert the string to a dictionary
    
    # Compute total weight sum (Triangular number formula) without adding it to the DataFrame
    total_weight_sum = num_edits * (num_edits + 1) / 2
    
    # Calculate base tracks per edit and extra tracks without adding them to the DataFrame
    base_tracks_per_edit = num_tracks // num_edits
    extra_tracks = num_tracks % num_edits
    
    track_distribution = [base_tracks_per_edit] * num_edits
    for i in range(extra_tracks):
        track_distribution[-(i + 1)] += 1  # Give extra tracks to most recent edits
    
    edit_weights = [(i + 1) / total_weight_sum for i in range(num_edits)]
    
    track_weights = []
    track_pos = 0  # Track index in the playlist
    
    for edit_idx, batch_size in enumerate(track_distribution):
        batch_weight = edit_weights[edit_idx]
        track_weight = batch_weight / batch_size  # Weight per track in batch
        
        # Round the track weight to 5 decimal places
        track_weight = round(track_weight, 5)
        
        for _ in range(batch_size):
            track_key = str(track_pos)  # Convert track_pos to string to match dictionary key format
            if track_key in tracks:  # Access track using string keys
                track_weights.append((tracks[track_key], track_weight))  # Add track and its weight
            track_pos += 1
    
    track_weights_list.append(track_weights)

# Add the list of track weights as a new column in the DataFrame
playlist['tracks_weights_5dp'] = track_weights_list

# Output the result
playlist[['num_tracks', 'num_edits', 'tracks_weights_5dp']].head()

,num_tracks,num_edits,tracks_weights_5dp
18539,32,11,"[(3689, 0.00758), (207774, 0.00758), (194775, ..."
18511,10,1,"[(160375, 0.1), (131195, 0.1), (164629, 0.1), ..."
18507,52,12,"[(104436, 0.00321), (229428, 0.00321), (25968,..."
6210,64,18,"[(244173, 0.00195), (210510, 0.00195), (9349, ..."
6245,73,4,"[(194410, 0.00556), (10513, 0.00556), (62267, ..."


Checking to ensure that the sum of the weights approximately equal to 1

In [128]:
# Calculate the sum of weights for each row
weights_sums = [sum(weight for _, weight in track_weights) for track_weights in playlist['tracks_weights_5dp']]

# Get the minimum and maximum sum of weights
min_weight_sum = min(weights_sums)
max_weight_sum = max(weights_sums)

# Print the range of the sum of weights
print(f"Range of weight sums: Min = {min_weight_sum}, Max = {max_weight_sum}")

Range of weight sums: Min = 0.9993599999999956, Max = 1.0008000000000015


# Duplicate the Playlist dataframe

As we have to capture the different means of the Unweighted and Weighted Tracks separately, we will duplicate the playlist

In [129]:
playlist_weighted = playlist.copy()

# Description of Notebook

A playlist is a collection of tracks. 

We will find these for each playlist:
- The aggregated features of the tracks (using Mean, Proportion, Centroid)
- The diversity of the playlist (using Variance, Shannon Entropy, Cosine Dissimilarity)

Note: 
- For the aggregated features, we will end up with 2 versions of aggregated playlist
    - One for unweighted tracks (by taking simple average)
    - The other for weighted tracks (by taking weighted average)
- For diversity calculation, we will only compute it on unweighted tracks.

# Proportions of `era` and `length`

Define a general function that will be used to calculate the proportion for each category later on

In [130]:
def compute_proportions(playlist, track, one_hot_columns, prefix):
    # Ensure track index column is a string for consistent mapping
    track['track_idx'] = track['track_idx'].astype(str)
    playlist['track_idx_list'] = playlist['track_idx_list'].apply(lambda lst: [str(x) for x in lst])

    # Convert True/False to 1/0 in the track DataFrame
    track[one_hot_columns] = track[one_hot_columns].astype(int)

    # Create a dictionary mapping track_idx to the one-hot encoded values
    track_dict = track.set_index('track_idx')[one_hot_columns].to_dict(orient='index')

    # Function to compute proportions for a playlist
    def compute_proportions_per_playlist(track_list):
        counts = {col: 0 for col in one_hot_columns}  # Initialize counts
        for track_idx in track_list:
            track_data = track_dict.get(track_idx, {col: 0 for col in one_hot_columns})  # Default to zeros
            for col, value in track_data.items():
                counts[col] += int(value)  # Convert boolean to integer and sum

        # Compute proportions
        total_tracks = len(track_list)
        if total_tracks == 0:
            return {col: 0 for col in one_hot_columns}  # Avoid division by zero
        
        return {col: counts[col] / total_tracks for col in one_hot_columns}

    # Apply the function to compute proportions
    playlist[f'{prefix}_proportions'] = playlist['track_idx_list'].apply(compute_proportions_per_playlist)

    # Create separate columns for proportions
    for col in one_hot_columns:
        col_name = prefix + '_' + col.replace(" ", "_").lower() + '_proportion'
        playlist[col_name] = playlist[f'{prefix}_proportions'].apply(lambda x: x[col])

    # Drop intermediate column
    playlist.drop(columns=[f'{prefix}_proportions'], inplace=True)

    return playlist

def compute_weighted_proportions(playlist, track, one_hot_columns, prefix):
    # Ensure track index column is a string for consistent mapping
    track['track_idx'] = track['track_idx'].astype(str)
    playlist['tracks_weights_5dp'] = playlist['tracks_weights_5dp'].apply(
        lambda lst: [(str(idx), weight) for idx, weight in lst]
    )

    # Convert True/False to 1/0 in the track DataFrame
    track[one_hot_columns] = track[one_hot_columns].astype(int)

    # Create a dictionary mapping track_idx to the one-hot encoded values
    track_dict = track.set_index('track_idx')[one_hot_columns].to_dict(orient='index')

    # Function to compute weighted proportions
    def compute_weighted_proportions_per_playlist(weighted_list):
        weighted_sums = {col: 0.0 for col in one_hot_columns}
        for track_idx, weight in weighted_list:
            track_data = track_dict.get(track_idx, {col: 0 for col in one_hot_columns})
            for col in one_hot_columns:
                weighted_sums[col] += track_data[col] * weight
        return weighted_sums

    # Apply the function
    playlist[f'{prefix}_proportions'] = playlist['tracks_weights_5dp'].apply(compute_weighted_proportions_per_playlist)

    # Expand into separate columns
    for col in one_hot_columns:
        col_name = prefix + '_' + col.replace(" ", "_").lower() + '_proportion'
        playlist[col_name] = playlist[f'{prefix}_proportions'].apply(lambda x: x[col])

    # Drop intermediate column
    playlist.drop(columns=[f'{prefix}_proportions'], inplace=True)

    return playlist

## Unweighted

### Proportion of `era`

In [131]:
era_columns = ["Early Years", "Classic Era", "Golden Era", "2000s", "Modern Era"]
playlist = compute_proportions(playlist, track, era_columns, 'era')

playlist[['playlist_idx'] + [col for col in playlist.columns if 'era' in col]].head()

,playlist_idx,era_early_years_proportion,era_classic_era_proportion,era_golden_era_proportion,era_2000s_proportion,era_modern_era_proportion
18539,1,0.0,0.000000,0.000000,0.000000,1.000000
18511,2,0.0,0.500000,0.200000,0.200000,0.100000
18507,3,0.0,0.134615,0.538462,0.288462,0.038462
6210,4,0.0,0.000000,0.000000,0.062500,0.937500
6245,5,0.0,0.000000,0.000000,0.013699,0.986301


### Proportion of `length`

In [132]:
length_columns = ["Short", "Medium", "Long"]
playlist = compute_proportions(playlist, track, length_columns, 'length')

playlist[['playlist_idx'] + [col for col in playlist.columns if 'length' in col]].head()

,playlist_idx,length_short_proportion,length_medium_proportion,length_long_proportion
18539,1,0.093750,0.718750,0.187500
18511,2,0.300000,0.400000,0.300000
18507,3,0.230769,0.519231,0.250000
6210,4,0.000000,0.375000,0.625000
6245,5,0.013699,0.547945,0.438356


## Weighted

### Proportion of `era`

In [133]:
era_columns = ["Early Years", "Classic Era", "Golden Era", "2000s", "Modern Era"]
playlist_weighted = compute_weighted_proportions(playlist_weighted, track, era_columns, 'era')

playlist_weighted[['playlist_idx'] + [col for col in playlist_weighted.columns if 'era' in col]].head()

,playlist_idx,era_early_years_proportion,era_classic_era_proportion,era_golden_era_proportion,era_2000s_proportion,era_modern_era_proportion
18539,1,0.0,0.00000,0.0000,0.00000,0.99997
18511,2,0.0,0.50000,0.2000,0.20000,0.10000
18507,3,0.0,0.12886,0.5616,0.28079,0.02885
6210,4,0.0,0.00000,0.0000,0.07310,0.92695
6245,5,0.0,0.00000,0.0000,0.02105,0.97902


### Proportion of `length`

In [134]:
length_columns = ["Short", "Medium", "Long"]
playlist_weighted = compute_weighted_proportions(playlist_weighted, track, length_columns, 'length')

playlist_weighted[['playlist_idx'] + [col for col in playlist_weighted.columns if 'length' in col]].head()

,playlist_idx,length_short_proportion,length_medium_proportion,length_long_proportion
18539,1,0.11616,0.76007,0.12374
18511,2,0.30000,0.40000,0.30000
18507,3,0.18208,0.53916,0.27886
6210,4,0.00000,0.32752,0.67253
6245,5,0.01111,0.59911,0.38985


# Diversity of `era`, `length` and `artist`

For diversity, we will only measure for the unweighted tracks

### Shannon Entropy for Categorical Variables

The **Shannon entropy** formula is used to measure the diversity of categorical variables in a dataset:

$$
H(X) = - \sum_{i=1}^{n} p(x_i) \cdot \log_2(p(x_i))
$$

Where:  
- $H(X)$ represents the diversity of a categorical variable (e.g., artist, era, length).  
- $p(x_i)$ represents the proportion of occurrences of category $x_i$ in the dataset.  
- $n$ is the total number of unique categories.  

---

### General Interpretation
- **Higher entropy** ($H(X)$  close to maximum): More balanced distribution of categories → Greater diversity.  <br><br>
- **Lower entropy** ($H(X)$ close to 0): The playlist is dominated by a small number of categories → Lower diversity.  
<br>

Entropy can be used to quantify the diversity of different categorical features in playlists.  

By computing entropy for **artist, era, and length**, we can quantify how varied a playlist is across different categorical dimensions.

In [135]:
def compute_shannon_entropy(proportions):
    proportions = np.array(proportions)
    # Filter out zero proportions to avoid log2(0)
    proportions = proportions[proportions > 0]
    entropy = -np.sum(proportions * np.log2(proportions))
    
    # Return 0 if entropy is close to 0
    if np.isclose(entropy, 0):
        return 0
    return entropy

def compute_entropy_for_playlist(playlist, category_columns, diversity_prefix):
    entropy_values = []
    diversity_column_name = f"{diversity_prefix}_diversity"

    # Apply entropy calculation for each playlist based on the specified category columns
    for index, row in playlist.iterrows():
        # Collect the proportions from the category columns for this row
        proportions = []
        for col in category_columns:
            # If the column contains dictionary-like data (e.g., era_proportions), extract proportions
            if isinstance(row[col], dict):
                proportions.extend(row[col].values())  # Extract values from the dictionary
            else:
                # If the column is already numeric, just add the proportion directly
                proportions.append(row[col])

        # Compute Shannon entropy if there are proportions to work with
        if proportions:
            entropy = compute_shannon_entropy(proportions)
            entropy_values.append(entropy)
        else:
            entropy_values.append(np.nan)

    # Ensure the length of entropy_values matches the number of rows in playlist
    if len(entropy_values) != len(playlist):
        raise ValueError("Length of entropy_values does not match the number of rows in the playlist.")

    # Add the entropy values as a new column
    playlist[diversity_column_name] = entropy_values
    return playlist

### Era Diversity
- Each track is categorized into "Early Years", "Classic Era", "Golden Era", "2000s", or "Modern Era".

- The entropy measures how evenly eras are distributed in the playlist.  

  - **High entropy** → The playlist contains a mix of different eras.  
  
  - **Low entropy** → The playlist is dominated by a single or few eras. 

In [136]:
era_proportion_columns = ["era_early_years_proportion", "era_classic_era_proportion", "era_golden_era_proportion", "era_2000s_proportion", "era_modern_era_proportion"]
playlist = compute_entropy_for_playlist(playlist, era_proportion_columns, 'era')
playlist[['playlist_idx', 'era_diversity']].head()

,playlist_idx,era_diversity
18539,1,0.000000
18511,2,1.760964
18507,3,1.568502
6210,4,0.337290
6245,5,0.104419


### Length Diversity
- Each track is categorized into "Short", "Medium", or "Long".  

- The entropy measures how balanced the playlist is across different track lengths.  

  - **High entropy** → A good mix of short, medium, and long tracks.  
  
  - **Low entropy** → The playlist consists mainly of one length category.  

In [137]:
length_proportion_columns = ["length_short_proportion", "length_medium_proportion", "length_long_proportion"]
playlist = compute_entropy_for_playlist(playlist, length_proportion_columns, 'length')
playlist[['playlist_idx', 'length_diversity']].head()

,playlist_idx,length_diversity
18539,1,1.115419
18511,2,1.570951
18507,3,1.479147
6210,4,0.954434
6245,5,1.081919


### Artist Diversity
- Each track in the playlist is mapped to an `artist_idx`.  

- We count how many tracks belong to each artist.  

- The entropy quantifies how evenly tracks are distributed across different artists.  

  - **High entropy** → Tracks are spread across many artists (high diversity).  
  
  - **Low entropy** → Most tracks belong to a few artists (low diversity).  

In [138]:
# Ensure track_idx in track dataframe is an integer
track['track_idx'] = track['track_idx'].astype(int)

# Create a dictionary mapping track_idx to artist_idx from the track dataframe
track_artist_dict = dict(zip(track['track_idx'].astype(str), track['artist_idx']))

# Apply this dictionary to the track_idx_list in the playlist DataFrame
playlist['artist_idx_list'] = playlist['track_idx_list'].apply(
    lambda track_list: [track_artist_dict.get(str(track_idx), np.nan) for track_idx in track_list]
)

# Remove NaN artist_idx values (if any)
playlist['artist_idx_list'] = playlist['artist_idx_list'].apply(
    lambda x: [artist for artist in x if not pd.isna(artist)]
)

# Initialize a list to store Shannon entropy values
shannon_entropy_values = []

# Compute Shannon entropy for each playlist
for artist_idx_list in playlist['artist_idx_list']:
    if artist_idx_list:  # Skip empty playlists
        # Count how many tracks each artist has in the playlist
        artist_counts = pd.Series(artist_idx_list).value_counts()

        # Calculate the proportion of tracks for each artist
        total_tracks = len(artist_idx_list)
        proportions = artist_counts / total_tracks

        # Compute Shannon entropy
        entropy = -np.sum(proportions * np.log2(proportions))

        shannon_entropy_values.append(entropy)
    else:
        shannon_entropy_values.append(np.nan)

# Add Shannon entropy values as a new column in the playlist dataframe
playlist['artist_diversity'] = shannon_entropy_values
playlist = playlist.drop(columns='artist_idx_list')

# Output the result
print(playlist[['playlist_idx', 'artist_diversity']].head())

       playlist_idx  artist_diversity
18539             1          4.937500
18511             2          3.321928
18507             3          5.508132
6210              4          4.994304
6245              5          4.701371


# Mean of `popularity`

The popularity mean will be used to represent the average popularity of tracks within a playlist. It helps to provide an overall sense of how popular the tracks are on average in the playlist.

## Unweighted

### Popularity Mean

In [139]:
# Ensure track_idx in track dataframe is an integer
track['track_idx'] = track['track_idx'].astype(int)

# Create a dictionary mapping track_idx to popularity from the track dataframe
track_popularity_dict = dict(zip(track['track_idx'].astype(str), track['track_popularity']))

# Apply this dictionary to the track_idx_list in the playlist DataFrame
playlist['track_popularity_list'] = playlist['track_idx_list'].apply(
    lambda track_list: [track_popularity_dict.get(str(track_idx), np.nan) for track_idx in track_list]
)

# Remove NaN popularity values (if any)
playlist['track_popularity_list'] = playlist['track_popularity_list'].apply(
    lambda x: [popularity for popularity in x if not pd.isna(popularity)]
)

# Initialize lists to store mean and variance values
popularity_mean_values = []

# Compute mean for each playlist (first chunk)
for popularity_list in playlist['track_popularity_list']:
    if popularity_list:  # Skip empty playlists
        # Compute mean for the track_popularity values in the playlist
        mean_popularity = np.mean(popularity_list)
        popularity_mean_values.append(mean_popularity)
    else:
        popularity_mean_values.append(np.nan)

# Add mean values as a new column
playlist['popularity_mean'] = popularity_mean_values

playlist[['playlist_idx', 'popularity_mean']].head()

,playlist_idx,popularity_mean
18539,1,42.625000
18511,2,49.300000
18507,3,47.923077
6210,4,43.968750
6245,5,48.643836


## Weighted

### Popularity Mean

In [140]:
# Ensure track_idx in track dataframe is int for consistency
track['track_idx'] = track['track_idx'].astype(int)

# Create a dictionary mapping track_idx (as str) to popularity
track_popularity_dict = dict(zip(track['track_idx'].astype(str), track['track_popularity']))

# Ensure tracks_weights_5dp uses str track_idx
playlist_weighted['tracks_weights_5dp'] = playlist_weighted['tracks_weights_5dp'].apply(
    lambda lst: [(str(idx), weight) for idx, weight in lst]
)

# Compute weighted mean popularity for each playlist
def weighted_mean_popularity(weighted_list):
    weighted_sum = 0.0
    total_weight = 0.0
    for track_idx, weight in weighted_list:
        popularity = track_popularity_dict.get(track_idx)
        if popularity is not None and not pd.isna(popularity):
            weighted_sum += popularity * weight
            total_weight += weight
    return weighted_sum / total_weight if total_weight > 0 else np.nan

playlist_weighted['popularity_mean'] = playlist_weighted['tracks_weights_5dp'].apply(weighted_mean_popularity)

# Preview
playlist_weighted[['playlist_idx', 'popularity_mean']].head()

,playlist_idx,popularity_mean
18539,1,38.441903
18511,2,49.300000
18507,3,47.394501
6210,4,46.348273
6245,5,50.498665


# Diversity of `Popularity`

The popularity variance will be used to assess the diversity or uniformity of the tracks in terms of popularity. A high variance indicates that there are tracks in the playlist with highly different popularity values, whereas a low variance suggests that most tracks have similar popularity.
<br> <br>
For example, a playlist (A) could have a popularity mean of 50, and popularity variance of 100. Whereas another playlist (B) could also have a popularity mean of 50, but the popularity variance is 10. From this, we can infer that playlist A does not have much consideration in the popularity of the tracks, whereas the playlist B consistently chooses tracks that have a similar popularity.
<br> <br>
For diversity, we will only measure for the unweighted tracks

Using the variance formula

In [141]:
popularity_variance_values = []

# Compute variance for each playlist (second chunk)
for popularity_list in playlist['track_popularity_list']:
    if popularity_list:  # Skip empty playlists
        # Compute variance for the track_popularity values in the playlist
        variance_popularity = np.var(popularity_list)
        popularity_variance_values.append(variance_popularity)
    else:
        popularity_variance_values.append(np.nan)

# Add variance values as a new column
playlist['popularity_diversity'] = popularity_variance_values
playlist = playlist.drop(columns='track_popularity_list')

playlist[['playlist_idx','popularity_diversity']].head()

,playlist_idx,popularity_diversity
18539,1,967.671875
18511,2,281.010000
18507,3,732.763314
6210,4,361.311523
6245,5,622.941640


# Centroid of `sentiment` and `genre`

To gain insight into the nature of each playlist, we will calculate the centroid of the sentiment and genre vectors based on the tracks it contains.

Define Functions 

In [142]:
# Function to compute centroids from the track_audio DataFrame and directly add to the playlist DataFrame
def compute_centroid(track_audio, playlist_dataframe, feature_columns, prefix):
    # Convert track_audio DataFrame to dictionary mapping track_idx to audio features
    track_audio_dict = {str(track_audio['track_idx'].iloc[i]): track_audio.iloc[i][feature_columns].values 
                        for i in range(len(track_audio))}
    
    # Store centroids in a separate list for playlist_recco
    playlist_centroids = []

    # Iterate through each playlist in playlist_dataframe
    for _, row in playlist_dataframe.iterrows():
        playlist_idx = row['playlist_idx']

        # Ensure track_idx_list is a list of track ids
        if isinstance(row['track_idx_list'], (list, np.ndarray)):
            track_ids = [str(idx).strip() for idx in row['track_idx_list']]
        else:
            print(f"Skipping Playlist {playlist_idx}: track_idx_list format not recognized.")
            continue

        # Convert track_idx to integer and filter None values
        track_ids_int = [int(idx) for idx in track_ids]
        
        # Extract audio feature vectors for tracks in the playlist (skip None values)
        playlist_vectors = np.array([track_audio_dict.get(str(idx)) for idx in track_ids_int if track_audio_dict.get(str(idx)) is not None])

        # Skip if no valid vectors found
        if playlist_vectors.size == 0:
            print(f"Skipping Playlist {playlist_idx} (No valid vectors found)")
            continue  

        # Compute the centroid (mean vector across audio feature space)
        centroid = np.mean(playlist_vectors, axis=0)

        # Store the centroid for playlist_recco
        playlist_centroids.append({
            'playlist_idx': playlist_idx,
            'genre_centroid': centroid
        })

    # Create a dictionary from the list of centroids for efficient lookup
    centroid_dict = {centroid['playlist_idx']: centroid['genre_centroid'] for centroid in playlist_centroids}

    # Merge the centroids back into the playlist dataframe
    playlist_dataframe[f'{prefix}_centroid'] = playlist_dataframe['playlist_idx'].map(centroid_dict)

    return playlist_dataframe

def compute_weighted_centroid(track_audio, playlist_dataframe, feature_columns, prefix):
    # Convert track_audio DataFrame to dictionary mapping track_idx to audio features
    track_audio_dict = {
        str(track_audio['track_idx'].iloc[i]): track_audio.iloc[i][feature_columns].values
        for i in range(len(track_audio))
    }

    # Store centroids in a separate list for playlist_recco
    playlist_centroids = []

    # Iterate through each playlist in playlist_dataframe
    for _, row in playlist_dataframe.iterrows():
        playlist_idx = row['playlist_idx']
        weighted_list = row.get('tracks_weights_5dp')

        # Validate the weighted list
        if not isinstance(weighted_list, list) or len(weighted_list) == 0:
            print(f"Skipping Playlist {playlist_idx}: Invalid or empty tracks_weights_5dp")
            continue

        weighted_sum = np.zeros(len(feature_columns))
        total_weight = 0.0
        missing_tracks = []

        for entry in weighted_list:
            # Ensure entry is a tuple of (track_idx, weight)
            if (not isinstance(entry, (list, tuple))) or len(entry) != 2:
                print(f"Skipping malformed entry in Playlist {playlist_idx}: {entry}")
                continue

            track_idx, weight = entry
            track_str = str(track_idx)
            vec = track_audio_dict.get(track_str)

            if vec is not None:
                weighted_sum += vec * weight
                total_weight += weight
            else:
                missing_tracks.append(track_idx)

        # Warn about any missing track vectors
        if missing_tracks:
            print(f"Missing vectors for track_idx {missing_tracks} in Playlist {playlist_idx}")

        # Only compute if total_weight > 0 (to avoid division by zero)
        if total_weight > 0:
            centroid = weighted_sum / total_weight
            playlist_centroids.append({
                'playlist_idx': playlist_idx,
                f'{prefix}_centroid': centroid
            })

    # Create a dictionary for centroids to map into playlist dataframe
    centroid_dict = {
        d['playlist_idx']: d[f'{prefix}_centroid']
        for d in playlist_centroids
    }

    # Merge centroids back into playlist dataframe
    playlist_dataframe[f'{prefix}_centroid'] = playlist_dataframe['playlist_idx'].map(centroid_dict)

    return playlist_dataframe

### Unweighted

We are taking the simple average for all the tracks

`sentiment` 

In [143]:
sentiment_columns = ["joy", "calm", "sadness", "fear", "energizing", "dreamy"]
playlist = compute_centroid(track, playlist, sentiment_columns, 'sentiment')
playlist[['playlist_idx', 'sentiment_centroid']].head()

,playlist_idx,sentiment_centroid
18539,1,"[0.09926200806814757, 0.10005938925602754, 0.1..."
18511,2,"[0.05571574525650652, 0.0415010311546112, 0.18..."
18507,3,"[0.08737024925150791, 0.0674074314898852, 0.17..."
6210,4,"[0.13178678037071967, 0.09436656111998172, 0.1..."
6245,5,"[0.09775544826018047, 0.045159551101370564, 0...."


`genre` 

In [144]:
genre_columns = [
    "Instrumental / Ambient Sounds", "Soft Acoustic / Classical", "Orchestral / Soundtrack",
    "Mid-tempo Pop / Indie", "Upbeat Electronic / Dance", "Slow & Melancholic (Sad Songs)",
    "Experimental / Jazz Fusion", "Lo-Fi / Chill Vibes"
]
playlist = compute_centroid(track, playlist, genre_columns, 'genre')
playlist[['playlist_idx', 'genre_centroid']].head()

,playlist_idx,genre_centroid
18539,1,"[0.3393935625613689, 0.060288508049334874, 0.1..."
18511,2,"[0.17539132513885045, 0.24404485650836025, 0.1..."
18507,3,"[0.1744458235077841, 0.0837604435839924, 0.239..."
6210,4,"[0.37840686770446474, 0.07747221948936736, 0.1..."
6245,5,"[0.47525405066733545, 0.029123384933868374, 0...."


### Weighted

We are taking the weighted average for all the tracks

`sentiment` 

In [145]:
sentiment_columns = ["joy", "calm", "sadness", "fear", "energizing", "dreamy"]
playlist_weighted = compute_weighted_centroid(track, playlist_weighted, sentiment_columns, 'sentiment')
playlist_weighted[['playlist_idx', 'sentiment_centroid']].head()

,playlist_idx,sentiment_centroid
18539,1,"[0.10064615769098226, 0.1018545425770113, 0.16..."
18511,2,"[0.05571574525650655, 0.041501031154611207, 0...."
18507,3,"[0.07943261941992363, 0.06375155027953708, 0.1..."
6210,4,"[0.12718252045315032, 0.09709627712204878, 0.1..."
6245,5,"[0.0993106944177796, 0.04668571631068851, 0.18..."


`genre` 

In [146]:
genre_columns = [
    "Instrumental / Ambient Sounds", "Soft Acoustic / Classical", "Orchestral / Soundtrack",
    "Mid-tempo Pop / Indie", "Upbeat Electronic / Dance", "Slow & Melancholic (Sad Songs)",
    "Experimental / Jazz Fusion", "Lo-Fi / Chill Vibes"
]
playlist_weighted = compute_weighted_centroid(track, playlist_weighted, genre_columns, 'genre')
playlist_weighted[['playlist_idx', 'genre_centroid']].head()

,playlist_idx,genre_centroid
18539,1,"[0.4161924381089197, 0.05407289159663813, 0.09..."
18511,2,"[0.1753913251388505, 0.2440448565083603, 0.104..."
18507,3,"[0.20508150982190532, 0.0771356619020066, 0.22..."
6210,4,"[0.372096392164391, 0.09668018027229801, 0.161..."
6245,5,"[0.49457423001006756, 0.025022362211549857, 0...."


# Diversity of `sentiment` and `genre`

To measure playlist diversity, we calculate the average cosine dissimilarity (1 - cosine similarity) of each track’s sentiment and genre vector from the playlist’s centroid, capturing how varied the tracks are relative to the overall theme.

For diversity, we will only measure for the unweighted tracks

Define Functions 

In [147]:
def cosine_similarity_to_centroid(track_audio, playlist_dataframe, feature_columns, prefix):
    # Convert track_audio DataFrame to dictionary mapping track_idx to audio features
    track_audio_dict = {str(track_audio['track_idx'].iloc[i]): track_audio.iloc[i][feature_columns].values 
                        for i in range(len(track_audio))}
    
    # Prepare result storage
    results = []
    
    # Store centroids in a separate list for playlist_recco
    playlist_centroids = []
    
    # Iterate through each playlist in playlist_dataframe
    for _, row in playlist_dataframe.iterrows():
        playlist_idx = row['playlist_idx']

        # Ensure track_idx_list is a list of track ids
        if isinstance(row['track_idx_list'], (list, np.ndarray)):
            track_ids = [str(idx).strip() for idx in row['track_idx_list']]
        else:
            print(f"Skipping Playlist {playlist_idx}: track_idx_list format not recognized.")
            continue

        # Convert track_idx to integer and filter None values
        track_ids_int = [int(idx) for idx in track_ids]
        
        # Extract audio feature vectors for tracks in the playlist (skip None values)
        playlist_vectors = []
        
        # Filter out track_ids that do not exist in track_audio_dict
        for idx in track_ids_int:
            track_features = track_audio_dict.get(str(idx))
            if track_features is not None:
                playlist_vectors.append(track_features)
            else:
                print(f"Track {idx} not found in track_audio_dict, skipping.")

        # Skip if no valid vectors found
        if not playlist_vectors:
            print(f"Skipping Playlist {playlist_idx} (No valid vectors found)")
            continue  

        # Convert playlist_vectors to a NumPy array
        playlist_vectors = np.array(playlist_vectors)

        # Compute the centroid (mean vector across audio feature space)
        centroid = np.mean(playlist_vectors, axis=0)
        
        # Store the centroid for playlist_recco
        playlist_centroids.append({
            'playlist_idx': playlist_idx,
            f'{prefix}_centroid': centroid
        })
        
        # Handle single-track playlists (assign perfect dissimilarity)
        if len(playlist_vectors) == 1:
            cos_similarities = np.array([1.0])  # Cosine similarity with itself is 1
        else:
            cos_similarities = cosine_similarity(playlist_vectors, centroid.reshape(1, -1)).flatten()

        # Store results (store 1 - similarity score for diversity interpretation)
        for i, track_id in enumerate(track_ids_int):
            # Ensure index is within bounds
            results.append({
                'playlist_idx': playlist_idx,
                'track_idx': track_id,
                f'{prefix}_diversity': float(1 - cos_similarities[i]),  # 1 - cosine similarity for diversity
            })

    # Convert results to a DataFrame
    results_df = pd.DataFrame(results)

    # Rename columns to lowercase with underscores instead of spaces
    results_df.columns = [col.replace(' ', '_').lower() for col in results_df.columns]

    # Group by playlist_idx and calculate the average diversity_score
    playlist_avg_diversity = results_df.groupby('playlist_idx')[f'{prefix}_diversity'].mean().reset_index()

    # Merge the average diversity score
    playlist_dataframe = pd.merge(playlist_dataframe, playlist_avg_diversity, on='playlist_idx', how='left')

    return playlist_dataframe

`sentiment`

In [148]:
playlist = cosine_similarity_to_centroid(track, playlist, sentiment_columns, 'sentiment')
playlist[['playlist_idx', 'sentiment_diversity']].head()

,playlist_idx,sentiment_diversity
0,1,0.039755
1,2,0.065906
2,3,0.096278
3,4,0.079647
4,5,0.091751


`genre`

In [149]:
playlist = cosine_similarity_to_centroid(track, playlist, genre_columns, 'genre')
playlist[['playlist_idx', 'genre_diversity']].head()

,playlist_idx,genre_diversity
0,1,0.444632
1,2,0.376064
2,3,0.395079
3,4,0.427553
4,5,0.388782


# Saving the features for playlist

### Unweighted Playlists

In [150]:
playlist = playlist.sort_values(by='playlist_idx', ascending=True).reset_index(drop=True)

columns_to_save = [
    'playlist_idx', 'num_tracks', 
    'track_idx_list', 'tracks_to_predict', 
    'num_edits', 'avg_tracks_per_edit', 'num_artists', 
    'popularity_mean',
    'era_early_years_proportion', 'era_classic_era_proportion', 'era_golden_era_proportion', 'era_2000s_proportion', 'era_modern_era_proportion',
    'length_short_proportion', 'length_medium_proportion', 'length_long_proportion',
    'sentiment_centroid',
    'genre_centroid'
]

df_to_save_unweighted = playlist[columns_to_save]
#df_to_save_unweighted.to_parquet('feature engineered datasets/playlist_feature_engineering_unweighted.parquet', index=False)
df_to_save_unweighted.to_parquet('playlist_feature_engineering_unweighted.parquet', index=False)

### Weighted Playlists

In [151]:
playlist_weighted = playlist_weighted.sort_values(by='playlist_idx', ascending=True).reset_index(drop=True)

df_to_save_weighted = playlist_weighted[columns_to_save]
#df_to_save_weighted.to_parquet('feature engineered datasets/playlist_feature_engineering_weighted.parquet', index=False)
df_to_save_weighted.to_parquet('playlist_feature_engineering_weighted.parquet', index=False)

### Dataset for Playlist Segmentation

In [152]:
columns_for_segmentation = [
    'playlist_idx', "era_diversity", "length_diversity", "artist_diversity", "popularity_diversity", "sentiment_diversity", "genre_diversity"
]

df_phase4 = playlist[columns_for_segmentation]
#df_phase4.to_parquet('../phase4_playlist_segmentation/dataset/playlist_diversity.parquet', index=False)
df_phase4.to_parquet('playlist_diversity.parquet', index=False)